# Pathology Model

This notebook trains and evaluates the `pathology` model. It reads the local biochemistry profile data, performs cleaning, applies clinical rules, splits the data, and runs model selection.

In [ ]:
# Imports and project path setup
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Set ROOT path relative to project structure
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.diagnostic_models import MODEL_SPECS, build_io_schema, candidate_pipelines, evaluate_pipeline, predict_for_backend, train_model


In [ ]:
# 1. Load Dataset
csv_path = ROOT / 'data' / 'real_processed' / 'nhanes_chronic_labs_real.csv'
df = pd.read_csv(csv_path)
display(df.head())
print('Shape:', df.shape)


In [ ]:
# 2. Cleaning and Labeling
spec = MODEL_SPECS['pathology']
df = df.dropna(subset=spec.numeric_features + ['sex']).copy()

def label_pathology(row):
    urgent = (
        row['fasting_glucose'] >= 250
        or row['hba1c'] >= 10
        or row['creatinine_mg_dl'] >= 2
        or row['alt_u_l'] >= 200
        or row['ast_u_l'] >= 200
        or row['total_cholesterol'] >= 320
    )
    follow_up = (
        row['fasting_glucose'] >= 126
        or row['hba1c'] >= 6.5
        or row['creatinine_mg_dl'] >= 1.3
        or row['alt_u_l'] >= 45
        or row['ast_u_l'] >= 45
        or row['total_cholesterol'] >= 240
    )
    return 'urgent_review' if urgent else 'follow_up' if follow_up else 'within_reference'

df['pathology_risk'] = df.apply(label_pathology, axis=1)
print(df['pathology_risk'].value_counts(normalize=True).round(3))


In [ ]:
# 3. Train / Validation / Test Split
X = df[spec.numeric_features + spec.categorical_features]
y = df[spec.target]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')


In [ ]:
# 4. Preprocessing + Model Selection
candidate_results = {}
fitted = {}
for candidate_name, pipeline in candidate_pipelines(spec).items():
    pipeline.fit(X_train, y_train)
    fitted[candidate_name] = pipeline
    candidate_results[candidate_name] = evaluate_pipeline(pipeline, X_val, y_val, spec)

display(pd.DataFrame(candidate_results).T.sort_values(['positive_recall', 'macro_f1'], ascending=False))
best_name = sorted(candidate_results, key=lambda k: (candidate_results[k]['positive_recall'], candidate_results[k]['macro_f1'], candidate_results[k]['accuracy']), reverse=True)[0]
pipeline = fitted[best_name]
print('Best model:', best_name)


In [ ]:
# 5. Evaluation Metrics
for split_name, X_split, y_split in [('validation', X_val, y_val), ('test', X_test, y_test)]:
    preds = pipeline.predict(X_split)
    print('\n' + split_name.upper())
    print(classification_report(y_split, preds, zero_division=0))
    print(confusion_matrix(y_split, preds, labels=pipeline.classes_))


In [ ]:
# 6. Save model
joblib.dump(pipeline, ROOT / 'models' / 'pathology_pipeline.joblib')
print('Model saved successfully.')
